# MPPP — 01 Process images (Navcam and Mastcam-Z) (MPPP 0.42.0)

1. **Select** PDS `.IMG` products: a named *scape*, a sol range, a waypoint radius, or an explicit list.
2. **Configure**: only differences from the defaults are needed; the full configuration is saved with the outputs.
3. **Process** → radiometrically normalised 16-bit PNGs (PDS names kept), reconstruction masks, per-image intrinsics and pose priors, a Metashape reference file, a COLMAP prior model, a manifest and a config snapshot.

**Metashape:** add the images, then *Import Reference* from `references.txt` (camera positions in local ENU metres, yaw/pitch/roll); masks are in the PNG alpha channel (or `masks/` with `write_mask_files`). **COLMAP:** `colmap/sparse_prior/` holds the prior model; `03_colmap_alignment.ipynb` builds the full database, rig and refined alignment.

The mask model is downloaded once on first use (about 125 MB, into the user cache) — see the README.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # Windows/conda: torch + numpy each loading Intel OpenMP; before torch
import sys, json
from pathlib import Path
# use the MPPP source this notebook belongs to (the folder above it that holds src/mppp), else an installed mppp
_here = Path.cwd().resolve()
_root = next((p for p in (_here, *_here.parents) if (p / "src" / "mppp").is_dir()), None)
if _root is not None:
    sys.path.insert(0, str(_root / "src"))
import mppp
print("MPPP", mppp.__version__, "|", Path(mppp.__file__).parent)

## 1  Inputs and outputs

In [ ]:
directory_input = Path("D:/data/m2020")          # your local PDS archive (searched recursively)
scapes_root     = Path("D:/scapes")              # one output folder per selection
# directory_input = Path(mppp.paths.source_root() / "tests" / "data" / "m20")   # the two example IMGs in the repository

# Waypoints (rover positions): the snapshot shipped with MPPP, or your refreshed copy in the user cache.
# refresh=True downloads the current file from the mission map server (do this for recent sols).
# waypoints = None works in each image's own site frame (no Mars lat/lon, stations not tied together).
waypoints = mppp.load_waypoints(refresh=False)
print(waypoints["_mppp_source"])

## 2  Select images
**Option A: a named scape** (groups 1 and 2; 110 mm Mastcam-Z frames dropped wherever a selection takes "all zooms").

| scape | sols | group 1 | group 2 |
|---|---|---|---|
| rockytop | 460-535 | NLF/NRF `_0A01` | FLF/FRF all, NLF/NRF all, ZL0/ZR0 `_034` |
| landing | 9-48 | — | NLF/NRF all, ZL0/ZR0 `_034` |
| belva | 770-835 | — | NLF/NRF all, ZL0/ZR0 all zooms except 110 |
| bunsen | 1055-1095 | — | NLF/NRF all, ZL0/ZR0 all zooms except 110 |
| hellandfjellet | 1601-1645 | — | NLF/NRF all, ZL0/ZR0 all zooms except 110 |

Options B–C cover anything else. Each option produces `filename_paths`, `directory_output` and `provenance`.

In [ ]:
from mppp.scapes import SCAPES, select_scape
from mppp.select import find_imgs, dedupe_by_version
from mppp.waypoints import waypoints_within_radius

# --- Option A: a named scape ----------------------------------------------------
scape = "belva"                        # rockytop | landing | belva | bunsen | hellandfjellet
filename_paths, provenance = select_scape(scape, directory_input, groups=(1, 2), exclude_zoom_mm=(110,))
directory_output = scapes_root / scape
print(json.dumps({k: provenance[k] for k in ("scape", "sols", "n_selected", "n_dropped_zoom",
                                             "n_superseded_versions", "by_camera_group")}, indent=1))

# --- Option B: a sol range (Navcam + Mastcam-Z 34 mm) --------------------------------
# filename_paths  = find_imgs(directory_input, ["NLF", "NRF"], sol_range=(709, 709))
# filename_paths += find_imgs(directory_input, ["ZL0", "ZR0"], sol_range=(709, 709), sequ_id="_034")
# filename_paths, _ = dedupe_by_version(filename_paths)            # highest product version; deletes nothing
# directory_output, provenance = scapes_root / "sol709", {"selection": "sols 709-709"}

# --- Option C: everything within `radius` metres of where the rover was on `anchor_sol`
# filename_paths = []
# for wp in waypoints_within_radius(waypoints, anchor_sol=464, radius=20):
#     p = wp["properties"]
#     filename_paths += find_imgs(directory_input, ["NLF", "NRF", "ZL0", "ZR0"], site_drive=(p["site"], p["drive"]))
# filename_paths, _ = dedupe_by_version(filename_paths)
# directory_output, provenance = scapes_root / "sol464_r20", {"selection": "waypoints within 20 m of sol 464"}

print(f"{len(filename_paths)} images -> {directory_output}")
for p in filename_paths[:5]: print("  ", p.name)

## 3  Configuration and mask model
`"checkpoint": "mppp_mask_v2"` is the released default model (`"mppp_mask_v1"` = the previous one) (downloaded once and checked against its SHA-256). A path to your own checkpoint (`.safetensors` or `.pt` + card) works too.

In [ ]:
from mppp.mask.hub import load_registry, model_path
# Before the model is published (or offline): MPPP installs the local copy from checkpoints_dir() automatically,
# or once by hand: from mppp.mask.hub import install_model; install_model(mppp.paths.checkpoints_dir() / "mppp_mask_convnext_tiny_s4_v2.safetensors")
reg = load_registry()
for name, e in reg["models"].items():
    print(f"{name}{' (default)' if name == reg['default'] else ''}: {e['file']}, val IoU {e.get('val_iou')}, "
          f"{'cached' if model_path(name).is_file() else 'downloaded on first use'}")

config = mppp.load_config({
    "masking": {"infer_mask": True, "checkpoint": "mppp_mask_v2"},
    "export":  {"formats": ["PNG16"]},          # add "PNG8" for an 8-bit copy (COLMAP feature extraction), "TIFF16" for TIFF
    # "resize": {"undistort": True},
})
print(json.dumps(config, indent=2))

## 4  Process
To drop unsuitable frames: process once, delete them from the output image folder, set `ONLY_REMAINING` to that format and run this cell again.

In [ ]:
# After a first run you can delete unsuitable images from the output folder (e.g. images_png16/) and rerun with
# ONLY_REMAINING = "PNG16": only the images still in that folder are processed again, and the manifest,
# references.txt and the COLMAP priors are built from them alone (nothing is deleted).
ONLY_REMAINING = None          # None | "PNG16" | "PNG8" | "TIFF16"
manifest = mppp.process_images(filename_paths, directory_output, config, waypoints, provenance=provenance,
                               only_existing=ONLY_REMAINING)
print(f"processed {manifest['n_processed']} / {manifest['n_requested']}; failed: {len(manifest['failed'])}")
for f in manifest["failed"]: print("  FAILED", f["file"], "->", f["error"])
if "warning" in manifest: print("WARNING:", manifest["warning"])

## 5  Inspect one result

In [ ]:
import matplotlib.pyplot as plt
im = mppp.MPPPImage(filename_paths[-1], config, waypoints)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(im.image_int8); ax[0].set_title(im.fn.stem, fontsize=8)
ax[1].imshow(im.mask, cmap="gray"); ax[1].set_title("reconstruction mask (white = include)")
for a in ax: a.axis("off")
print("\n".join(im.log)); print(im.intrinsics.to_dict()); print(im.pose.to_dict()); print(im.geo)